# HortiAmiga
## Asistente inteligente para la orientación y cuidado de hortalizas en huertos urbanos

Prototipo académico de chatbot basado en reglas, desarrollado con Python 3.10 o superior y NLTK. Está dirigido a personas principiantes que desean iniciar o cuidar un huerto doméstico en balcones, patios, azoteas, ventanas o macetas.

**Alcance:** orientación inicial sobre cultivos comunes, temporada, riego, iluminación, sustrato y problemas visibles frecuentes. Las respuestas son educativas y no sustituyen la valoración de una persona especialista.

## 1. Preparación del entorno

Esta celda instala NLTK cuando el notebook se ejecuta en Google Colab.

In [1]:
!pip install nltk pandas -q


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import re
import unicodedata
import pandas as pd
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

tokenizer = RegexpTokenizer(r"\w+")
stemmer = SnowballStemmer("spanish")

print("Entorno de HortiAmiga preparado correctamente.")

Entorno de HortiAmiga preparado correctamente.


## 2. Preprocesamiento con NLTK

El texto se convierte a minúsculas, se eliminan acentos y signos, se tokeniza y se reduce cada palabra a su raíz mediante stemming. También se retiran palabras muy frecuentes que no ayudan a distinguir intenciones.

In [3]:
PALABRAS_VACIAS = {
    "a", "al", "algo", "como", "con", "cual", "cuando", "de", "del",
    "el", "en", "es", "esta", "este", "la", "las", "lo", "los", "me",
    "mi", "para", "por", "puedo", "que", "quiero", "se", "su", "tengo",
    "un", "una", "y"
}

def normalizar_texto(texto):
    """Convierte el texto a una forma uniforme para compararlo."""
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    texto = re.sub(r"[^a-z0-9\s]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()

def tokenizar_y_stem(texto, quitar_vacias=True):
    """Tokeniza con NLTK y aplica stemming en español."""
    tokens = tokenizer.tokenize(normalizar_texto(texto))
    if quitar_vacias:
        tokens = [t for t in tokens if t not in PALABRAS_VACIAS]
    stems = [stemmer.stem(token) for token in tokens]
    return tokens, stems

print(normalizar_texto("¿Cuánta luz necesita el JITOMATE?"))
print(tokenizar_y_stem("¿Cada cuánto debo regar mis lechugas?"))

cuanta luz necesita el jitomate
(['cada', 'cuanto', 'debo', 'regar', 'mis', 'lechugas'], ['cad', 'cuant', 'deb', 'reg', 'mis', 'lechug'])


## 3. Base de conocimiento

La información se mantiene separada de las frases usadas para detectar intenciones. Esto facilita ampliar o corregir el conocimiento sin modificar el motor conversacional.

In [4]:
base_conocimiento = {
    "jitomate": {
        "temporada": "primavera y comienzos de verano, evitando frío intenso",
        "riego": "riego regular de 2 a 4 veces por semana; revisa que el sustrato esté ligeramente húmedo y sin encharcamientos",
        "luz": "entre 6 y 8 horas de sol directo",
        "sustrato": "suelto, fértil, con composta y buen drenaje"
    },
    "chile": {
        "temporada": "primavera, cuando disminuye el riesgo de temperaturas bajas",
        "riego": "riego moderado de 2 a 3 veces por semana, ajustado al calor y al tamaño de la maceta",
        "luz": "al menos 6 horas de sol directo",
        "sustrato": "fértil, aireado y con drenaje eficiente"
    },
    "lechuga": {
        "temporada": "otoño, invierno o primavera en climas templados",
        "riego": "riego ligero y frecuente para mantener humedad uniforme, sin saturar la tierra",
        "luz": "de 4 a 6 horas de luz; agradece semisombra cuando hace mucho calor",
        "sustrato": "rico en materia orgánica y capaz de conservar humedad sin encharcarse"
    },
    "zanahoria": {
        "temporada": "otoño a primavera en climas templados",
        "riego": "riego ligero y constante para evitar que el suelo se seque por completo",
        "luz": "entre 4 y 6 horas de sol",
        "sustrato": "profundo, suelto y sin piedras para permitir el crecimiento de la raíz"
    },
    "rabano": {
        "temporada": "otoño, invierno o primavera; evita periodos de calor extremo",
        "riego": "riego ligero y regular para mantener la humedad",
        "luz": "de 4 a 6 horas de sol",
        "sustrato": "suelto, ligero y con buen drenaje"
    },
    "cebolla": {
        "temporada": "otoño o invierno en zonas templadas, según la variedad",
        "riego": "riego moderado; reduce la frecuencia cuando el bulbo se acerque a la madurez",
        "luz": "al menos 6 horas de sol directo",
        "sustrato": "suelto, fértil y bien drenado"
    },
    "cilantro": {
        "temporada": "otoño a primavera; el calor intenso acelera su floración",
        "riego": "riego ligero de 2 a 3 veces por semana, según la temperatura",
        "luz": "entre 4 y 6 horas de luz, con semisombra en clima caluroso",
        "sustrato": "ligero, con materia orgánica y buen drenaje"
    },
    "albahaca": {
        "temporada": "primavera y verano",
        "riego": "riego regular cuando la superficie del sustrato empiece a secarse",
        "luz": "entre 5 y 7 horas de luz o sol suave",
        "sustrato": "fértil, aireado y con buen drenaje"
    },
    "menta": {
        "temporada": "primavera y otoño, aunque puede mantenerse gran parte del año en clima templado",
        "riego": "riego frecuente para conservar el sustrato ligeramente húmedo",
        "luz": "luz abundante o semisombra; evita el sol fuerte de la tarde",
        "sustrato": "rico en materia orgánica y con buena retención de humedad"
    }
}

recomendaciones_espacio = {
    "balcon": ["lechuga", "rabano", "cilantro", "albahaca"],
    "ventana": ["cilantro", "albahaca", "menta"],
    "patio": ["jitomate", "chile", "lechuga", "zanahoria"],
    "azotea": ["jitomate", "chile", "cebolla", "rabano"],
    "maceta": ["lechuga", "rabano", "cilantro", "albahaca"]
}

problemas_frecuentes = {
    "hojas amarillas": "pueden relacionarse con exceso de riego, drenaje deficiente o falta de nutrientes. Revisa la humedad antes de volver a regar",
    "manchas": "pueden aparecer por hongos, quemadura solar o daño de plagas. Evita mojar las hojas por la noche y observa si las manchas se extienden",
    "marchitez": "revisa primero la humedad del sustrato y la exposición al calor. Tanto la falta como el exceso de agua pueden causar marchitez",
    "crecimiento lento": "revisa horas de luz, tamaño de la maceta, drenaje y disponibilidad de nutrientes",
    "pulgon": "retira los insectos con agua a presión suave y revisa el envés de las hojas. Aísla la planta si la presencia aumenta",
    "mosca blanca": "revisa el envés de las hojas, retira las más afectadas y utiliza trampas adhesivas amarillas como medida de monitoreo",
    "cochinilla": "retira manualmente los individuos visibles y separa temporalmente la planta de las demás"
}

print(f"Cultivos disponibles: {len(base_conocimiento)}")

Cultivos disponibles: 9


## 4. Intenciones conversacionales

Cada intención incluye frases de ejemplo, palabras clave distintivas y una respuesta base. Las palabras clave tienen mayor peso que las coincidencias generales.

In [5]:
intenciones = {
    "saludo": {
        "ejemplos": ["hola", "buenos dias", "buenas tardes", "que tal hortiamiga", "necesito orientacion"],
        "claves": ["hola", "buenos dias", "buenas tardes", "buenas noches", "saludos"],
        "respuesta": "¡Hola! Soy HortiAmiga. Puedo orientarte para iniciar un huerto y resolver dudas sobre hortalizas, temporada, riego, luz, sustrato y problemas frecuentes."
    },
    "iniciar_huerto": {
        "ejemplos": ["quiero iniciar un huerto", "como empiezo mi huerto", "que necesito para comenzar", "primeros pasos para cultivar", "quiero hacer un huerto en casa"],
        "claves": ["iniciar huerto", "empezar huerto", "comenzar huerto", "primeros pasos", "hacer huerto"],
        "respuesta": "Para comenzar, identifica el espacio disponible, calcula sus horas de luz, elige recipientes con drenaje y empieza con pocos cultivos. Después podemos elegir hortalizas adecuadas para tu espacio."
    },
    "recomendar_hortaliza": {
        "ejemplos": ["que puedo sembrar en mi balcon", "recomiendame una hortaliza", "que cultivo cabe en una maceta", "que plantar en poco espacio", "cultivos para una azotea"],
        "claves": ["que sembrar", "que plantar", "que cultivar", "recomienda", "recomiendame", "cultivos para", "hortalizas para"],
        "respuesta": "Puedo recomendarte cultivos si me indicas si utilizarás balcón, ventana, patio, azotea o maceta. También ayuda conocer las horas de luz."
    },
    "consultar_temporada": {
        "ejemplos": ["cuando se siembra el jitomate", "en que temporada planto rabano", "fecha para cultivar lechuga", "puedo sembrar en invierno", "mejor epoca para el cilantro"],
        "claves": ["temporada", "epoca", "cuando sembrar", "cuando plantar", "fecha de siembra", "primavera", "verano", "otono", "invierno"],
        "respuesta": "Indícame el cultivo y te compartiré una temporada general. La fecha exacta puede variar según el clima de tu localidad."
    },
    "consultar_riego": {
        "ejemplos": ["cada cuanto riego el jitomate", "cuanta agua necesita la lechuga", "frecuencia de riego", "debo regar diario", "como regar mi albahaca"],
        "claves": ["riego", "regar", "agua", "humedad", "cada cuanto"],
        "respuesta": "Dime qué cultivo tienes para darte una orientación de riego. Comprueba siempre la humedad del sustrato porque el clima y la maceta modifican la frecuencia."
    },
    "consultar_iluminacion": {
        "ejemplos": ["cuanto sol necesita el chile", "horas de luz para cilantro", "puede estar en sombra", "necesita sol directo", "sirve la semisombra"],
        "claves": ["luz", "sol", "sombra", "semisombra", "iluminacion", "horas de sol"],
        "respuesta": "Indícame el cultivo para orientarte sobre sus necesidades de luz. Observa también el sol real que recibe tu espacio durante el día."
    },
    "consultar_sustrato": {
        "ejemplos": ["que tierra necesita la zanahoria", "sustrato para jitomate", "que mezcla uso en la maceta", "necesita composta", "como preparar la tierra"],
        "claves": ["sustrato", "tierra", "composta", "abono", "mezcla", "drenaje"],
        "respuesta": "Dime qué deseas cultivar. En general, las macetas necesitan un sustrato aireado, materia orgánica y orificios que permitan drenar el exceso de agua."
    },
    "consultar_problema": {
        "ejemplos": ["mi planta tiene hojas amarillas", "aparecieron manchas en las hojas", "mi lechuga esta marchita", "hay pulgones en el chile", "la planta crece muy lento"],
        "claves": ["hojas amarillas", "manchas", "marchita", "marchitez", "pulgon", "mosca blanca", "cochinilla", "plaga", "crecimiento lento", "no crece"],
        "respuesta": "Describe el síntoma visible y, si es posible, indica el cultivo. Te daré una orientación preventiva, no un diagnóstico definitivo."
    },
    "despedida": {
        "ejemplos": ["gracias", "hasta luego", "adios", "eso es todo", "nos vemos"],
        "claves": ["gracias", "hasta luego", "adios", "nos vemos", "eso es todo"],
        "respuesta": "¡Con gusto! Espero que tu huerto siga creciendo. Hasta pronto."
    }
}

## 5. Vocabulario y detección de intención

El puntaje combina coincidencias con las frases de ejemplo y coincidencias con palabras clave. Se exige evidencia mínima; cuando no existe, se activa el fallback.

In [6]:
def construir_vocabularios(datos_intenciones):
    vocabularios = {}
    claves_normalizadas = {}

    for nombre, datos in datos_intenciones.items():
        stems = set()
        for ejemplo in datos["ejemplos"]:
            _, stems_ejemplo = tokenizar_y_stem(ejemplo)
            stems.update(stems_ejemplo)
        vocabularios[nombre] = stems
        claves_normalizadas[nombre] = [normalizar_texto(c) for c in datos["claves"]]

    return vocabularios, claves_normalizadas

vocabularios, claves_normalizadas = construir_vocabularios(intenciones)

def detectar_intencion(mensaje, detalle=False):
    texto = normalizar_texto(mensaje)
    _, stems = tokenizar_y_stem(texto)
    stems_mensaje = set(stems)
    puntajes = {}

    for nombre in intenciones:
        coincidencias = len(stems_mensaje.intersection(vocabularios[nombre]))
        coincidencias_clave = sum(
            1 for clave in claves_normalizadas[nombre]
            if re.search(rf"\b{re.escape(clave)}\b", texto)
        )
        puntajes[nombre] = coincidencias + (3 * coincidencias_clave)

    mejor = max(puntajes, key=puntajes.get)
    mejor_puntaje = puntajes[mejor]

    # Una palabra de vocabulario aislada no es evidencia suficiente.
    if mejor_puntaje < 2:
        mejor, mejor_puntaje = "fallback", 0

    if detalle:
        return mejor, mejor_puntaje, puntajes
    return mejor, mejor_puntaje

print(detectar_intencion("¿Cada cuánto debo regar el cilantro?", detalle=True))
print(detectar_intencion("Quiero cambiar mi correo electrónico", detalle=True))

('consultar_riego', 10, {'saludo': 0, 'iniciar_huerto': 0, 'recomendar_hortaliza': 0, 'consultar_temporada': 1, 'consultar_riego': 10, 'consultar_iluminacion': 2, 'consultar_sustrato': 0, 'consultar_problema': 0, 'despedida': 0})
('fallback', 0, {'saludo': 0, 'iniciar_huerto': 0, 'recomendar_hortaliza': 0, 'consultar_temporada': 0, 'consultar_riego': 0, 'consultar_iluminacion': 0, 'consultar_sustrato': 0, 'consultar_problema': 0, 'despedida': 0})


## 6. Detección de entidades

Las entidades se reconocen mediante diccionarios y expresiones regulares. El prototipo identifica cultivos, espacios, condiciones de iluminación y síntomas.

In [7]:
catalogo_entidades = {
    "cultivo": {
        "jitomate": ["jitomate", "tomate"],
        "chile": ["chile", "chiles"],
        "lechuga": ["lechuga", "lechugas"],
        "zanahoria": ["zanahoria", "zanahorias"],
        "rabano": ["rabano", "rabanos"],
        "cebolla": ["cebolla", "cebollas"],
        "cilantro": ["cilantro"],
        "albahaca": ["albahaca"],
        "menta": ["menta", "hierbabuena"]
    },
    "espacio": {
        "balcon": ["balcon"],
        "ventana": ["ventana", "repisa"],
        "patio": ["patio"],
        "azotea": ["azotea", "terraza"],
        "maceta": ["maceta", "macetas", "jardinera"]
    },
    "iluminacion": {
        "sol directo": ["sol directo", "mucho sol"],
        "semisombra": ["semisombra", "media sombra", "poco sol"],
        "sombra": ["sombra", "sin sol"]
    },
    "problema": {
        "hojas amarillas": ["hojas amarillas", "hoja amarilla", "amarillentas"],
        "manchas": ["manchas", "mancha en las hojas"],
        "marchitez": ["marchita", "marchito", "marchitez", "decaida"],
        "crecimiento lento": ["crecimiento lento", "no crece", "crece poco"],
        "pulgon": ["pulgon", "pulgones"],
        "mosca blanca": ["mosca blanca"],
        "cochinilla": ["cochinilla", "cochinillas"]
    }
}

def detectar_entidades(mensaje):
    texto = normalizar_texto(mensaje)
    entidades = {}

    for tipo, valores in catalogo_entidades.items():
        for valor, alias in valores.items():
            if any(re.search(rf"\b{re.escape(normalizar_texto(a))}\b", texto) for a in alias):
                entidades[tipo] = valor
                break

    return entidades

print(detectar_entidades("Mi jitomate tiene hojas amarillas en la maceta del balcón"))

{'cultivo': 'jitomate', 'espacio': 'balcon', 'problema': 'hojas amarillas'}


## 7. Generación de respuestas

La respuesta se construye con la intención detectada, las entidades y la base de conocimiento.

In [8]:
def generar_respuesta(mensaje):
    intencion, puntaje = detectar_intencion(mensaje)
    entidades = detectar_entidades(mensaje)
    cultivo = entidades.get("cultivo")
    espacio = entidades.get("espacio")
    problema = entidades.get("problema")

    if intencion == "fallback":
        respuesta = (
            "No estoy segura de haber entendido. Puedes preguntarme cómo iniciar un huerto, "
            "qué hortaliza cultivar o sobre temporada, riego, luz, sustrato y problemas frecuentes."
        )
    elif intencion == "recomendar_hortaliza" and espacio:
        opciones = recomendaciones_espacio.get(espacio, [])
        respuesta = (
            f"Para un espacio tipo {espacio} puedes considerar: {', '.join(opciones)}. "
            "Antes de elegir, confirma cuántas horas de luz recibe el lugar."
        )
    elif intencion == "recomendar_hortaliza" and cultivo:
        respuesta = (
            f"{cultivo.capitalize()} puede cultivarse en un huerto urbano. "
            f"Necesita {base_conocimiento[cultivo]['luz']} y un sustrato {base_conocimiento[cultivo]['sustrato']}."
        )
    elif intencion == "consultar_temporada" and cultivo:
        respuesta = (
            f"La temporada general para {cultivo} es {base_conocimiento[cultivo]['temporada']}. "
            "Ajústala al clima de tu localidad."
        )
    elif intencion == "consultar_riego" and cultivo:
        respuesta = f"Para {cultivo} se recomienda {base_conocimiento[cultivo]['riego']}."
    elif intencion == "consultar_iluminacion" and cultivo:
        respuesta = f"El cultivo de {cultivo} requiere {base_conocimiento[cultivo]['luz']}."
    elif intencion == "consultar_sustrato" and cultivo:
        respuesta = f"Para {cultivo} utiliza un sustrato {base_conocimiento[cultivo]['sustrato']}."
    elif intencion == "consultar_problema" and problema:
        prefijo = f"En tu {cultivo}, " if cultivo else "Como orientación inicial, "
        respuesta = prefijo + problemas_frecuentes[problema] + ". Observa la evolución antes de aplicar tratamientos."
    else:
        respuesta = intenciones[intencion]["respuesta"]

    return {
        "mensaje": mensaje,
        "intencion": intencion,
        "puntaje": puntaje,
        "entidades": entidades,
        "respuesta": respuesta
    }

mensajes_demostracion = [
    "Hola HortiAmiga",
    "¿Qué puedo sembrar en mi balcón?",
    "¿Cada cuánto riego el jitomate?",
    "Mi albahaca tiene hojas amarillas",
    "Quiero cambiar mi correo electrónico"
]

for mensaje in mensajes_demostracion:
    resultado = generar_respuesta(mensaje)
    print(f"Usuario: {mensaje}")
    print(f"Intención: {resultado['intencion']} | Entidades: {resultado['entidades']}")
    print(f"HortiAmiga: {resultado['respuesta']}\n")

Usuario: Hola HortiAmiga
Intención: saludo | Entidades: {}
HortiAmiga: ¡Hola! Soy HortiAmiga. Puedo orientarte para iniciar un huerto y resolver dudas sobre hortalizas, temporada, riego, luz, sustrato y problemas frecuentes.

Usuario: ¿Qué puedo sembrar en mi balcón?
Intención: recomendar_hortaliza | Entidades: {'espacio': 'balcon'}
HortiAmiga: Para un espacio tipo balcon puedes considerar: lechuga, rabano, cilantro, albahaca. Antes de elegir, confirma cuántas horas de luz recibe el lugar.

Usuario: ¿Cada cuánto riego el jitomate?
Intención: consultar_riego | Entidades: {'cultivo': 'jitomate'}
HortiAmiga: Para jitomate se recomienda riego regular de 2 a 4 veces por semana; revisa que el sustrato esté ligeramente húmedo y sin encharcamientos.

Usuario: Mi albahaca tiene hojas amarillas
Intención: consultar_problema | Entidades: {'cultivo': 'albahaca', 'problema': 'hojas amarillas'}
HortiAmiga: En tu albahaca, pueden relacionarse con exceso de riego, drenaje deficiente o falta de nutrien

## 8. Conversación interactiva

Ejecuta la siguiente celda en Colab y escribe mensajes. Para terminar, escribe salir. La función devuelve el historial como una tabla.

In [9]:
def iniciar_chatbot():
    historial = []
    print("HortiAmiga: ¡Hola! Puedo ayudarte con tu huerto urbano.")
    print("HortiAmiga: Escribe 'salir' para terminar.\n")

    while True:
        mensaje = input("Usuario: ").strip()
        if normalizar_texto(mensaje) == "salir":
            print("HortiAmiga: ¡Hasta pronto! Que tu huerto siga creciendo.")
            break
        if not mensaje:
            print("HortiAmiga: Escribe una pregunta para poder orientarte.\n")
            continue

        resultado = generar_respuesta(mensaje)
        historial.append(resultado)
        print(f"HortiAmiga: {resultado['respuesta']}\n")

    return pd.DataFrame(historial)

# Descomenta las siguientes líneas para conversar en Colab:
#historial_chat = iniciar_chatbot()
#historial_chat

## 9. Pruebas funcionales

Se evalúan 15 mensajes distintos. Incluyen todas las intenciones, variaciones no idénticas a los ejemplos y dos consultas fuera del alcance.

In [10]:
casos_prueba = [
    {"mensaje": "Hola, buenos días", "esperada": "saludo"},
    {"mensaje": "Deseo comenzar un huerto en casa", "esperada": "iniciar_huerto"},
    {"mensaje": "¿Qué puedo cultivar en un balcón?", "esperada": "recomendar_hortaliza"},
    {"mensaje": "Recomiéndame una hortaliza para maceta", "esperada": "recomendar_hortaliza"},
    {"mensaje": "¿Cuándo debo sembrar jitomate?", "esperada": "consultar_temporada"},
    {"mensaje": "¿En qué temporada crece mejor el rábano?", "esperada": "consultar_temporada"},
    {"mensaje": "¿Cada cuánto debo regar la lechuga?", "esperada": "consultar_riego"},
    {"mensaje": "¿Cuánta agua necesita mi albahaca?", "esperada": "consultar_riego"},
    {"mensaje": "¿Cuántas horas de sol necesita el cilantro?", "esperada": "consultar_iluminacion"},
    {"mensaje": "¿Qué tierra uso para cultivar zanahoria?", "esperada": "consultar_sustrato"},
    {"mensaje": "Mi albahaca tiene las hojas amarillas", "esperada": "consultar_problema"},
    {"mensaje": "Encontré pulgones en mi chile", "esperada": "consultar_problema"},
    {"mensaje": "Gracias, eso es todo", "esperada": "despedida"},
    {"mensaje": "Quiero cambiar mi correo electrónico", "esperada": "fallback"},
    {"mensaje": "¿Quién ganó el partido de futbol?", "esperada": "fallback"}
]

resultados_prueba = []
for numero, caso in enumerate(casos_prueba, start=1):
    resultado = generar_respuesta(caso["mensaje"])
    resultados_prueba.append({
        "caso": numero,
        "mensaje": caso["mensaje"],
        "intencion_esperada": caso["esperada"],
        "intencion_detectada": resultado["intencion"],
        "correcto": caso["esperada"] == resultado["intencion"],
        "entidades": resultado["entidades"],
        "respuesta": resultado["respuesta"]
    })

df_resultados = pd.DataFrame(resultados_prueba)
df_resultados

,caso,mensaje,intencion_esperada,intencion_detectada,correcto,entidades,respuesta
0,1,"Hola, buenos días",saludo,saludo,True,{},¡Hola! Soy HortiAmiga. Puedo orientarte para i...
1,2,Deseo comenzar un huerto en casa,iniciar_huerto,iniciar_huerto,True,{},"Para comenzar, identifica el espacio disponibl..."
2,3,¿Qué puedo cultivar en un balcón?,recomendar_hortaliza,recomendar_hortaliza,True,{'espacio': 'balcon'},Para un espacio tipo balcon puedes considerar:...
3,4,Recomiéndame una hortaliza para maceta,recomendar_hortaliza,recomendar_hortaliza,True,{'espacio': 'maceta'},Para un espacio tipo maceta puedes considerar:...
4,5,¿Cuándo debo sembrar jitomate?,consultar_temporada,consultar_temporada,True,{'cultivo': 'jitomate'},La temporada general para jitomate es primaver...
5,6,¿En qué temporada crece mejor el rábano?,consultar_temporada,consultar_temporada,True,{'cultivo': 'rabano'},"La temporada general para rabano es otoño, inv..."
6,7,¿Cada cuánto debo regar la lechuga?,consultar_riego,consultar_riego,True,{'cultivo': 'lechuga'},Para lechuga se recomienda riego ligero y frec...
7,8,¿Cuánta agua necesita mi albahaca?,consultar_riego,consultar_riego,True,{'cultivo': 'albahaca'},Para albahaca se recomienda riego regular cuan...
8,9,¿Cuántas horas de sol necesita el cilantro?,consultar_iluminacion,consultar_iluminacion,True,{'cultivo': 'cilantro'},El cultivo de cilantro requiere entre 4 y 6 ho...
9,10,¿Qué tierra uso para cultivar zanahoria?,consultar_sustrato,consultar_sustrato,True,{'cultivo': 'zanahoria'},"Para zanahoria utiliza un sustrato profundo, s..."


In [11]:
precision = df_resultados["correcto"].mean()
tasa_fallback = (df_resultados["intencion_detectada"] == "fallback").mean()
aciertos = int(df_resultados["correcto"].sum())

print("RESULTADOS DE LA EVALUACIÓN")
print(f"Casos evaluados: {len(df_resultados)}")
print(f"Aciertos: {aciertos}")
print(f"Errores: {len(df_resultados) - aciertos}")
print(f"Precisión de intención: {precision:.2%}")
print(f"Tasa de fallback: {tasa_fallback:.2%}")

if not df_resultados["correcto"].all():
    print("\nCasos que requieren revisión:")
    display(df_resultados.loc[~df_resultados["correcto"], [
        "mensaje", "intencion_esperada", "intencion_detectada"
    ]])

RESULTADOS DE LA EVALUACIÓN
Casos evaluados: 15
Aciertos: 15
Errores: 0
Precisión de intención: 100.00%
Tasa de fallback: 13.33%


## 10. Limitaciones de esta versión

- El sistema depende de reglas, ejemplos y palabras clave definidos manualmente.
- No conserva contexto entre turnos; cada mensaje se analiza de manera independiente.
- Las recomendaciones son generales y no consideran todavía ubicación, clima, fecha exacta ni pronóstico meteorológico.
- La detección de síntomas ofrece orientación preventiva y no identifica enfermedades con certeza.
- El catálogo inicial se limita a nueve cultivos comunes.
- Los errores ortográficos severos y expresiones muy diferentes a los ejemplos pueden activar el fallback.

## Mejoras propuestas

En etapas posteriores se puede incorporar clasificación supervisada de texto, memoria conversacional, ubicación y calendario de siembra, reconocimiento de fotografías de hojas, una base de conocimiento ampliada y una interfaz web accesible.